# LLM fine-tuning starter

Load the textified split and run a short SFT pass with Hugging Face
`Trainer` or a LoRA adapter on a small base model. This notebook is
intentionally minimal — it shows loading + formatting, not a full
training recipe.

In [ ]:
from datasets import load_dataset

ds = load_dataset('twelvedata/financial-world-model', 'text_1day')
print(ds)
print(ds['train'][0])

In [ ]:
SYSTEM = (
    'You are a careful market analyst. Given a single bar of price '
    'and indicator data, predict the sign of the next bar log return.'
)

def format_example(row):
    return {
        'text': (
            f'<|system|>\n{SYSTEM}\n<|user|>\n{row["prompt"]}\n'
            f'<|assistant|>\n{row["label"]}\n'
        )
    }

train = ds['train'].map(format_example, remove_columns=ds['train'].column_names)
val = ds['val'].map(format_example, remove_columns=ds['val'].column_names)
train[0]

From here, plug into your preferred fine-tuning stack
(`trl.SFTTrainer`, `peft` LoRA, `unsloth`, etc.). Evaluate on
`ds['test']` — note that `test` begins 2025-01-01 in the default
split config.